# Unified Classifier Notebook (Training‑Only) — XGBoost vs LightGBM vs CatBoost

This notebook **trains** three state‑of‑the‑art Gradient Boosted Decision Tree (GBDT) classifiers on **training CSV datasets**.

Models:
- XGBoost (primary)
- LightGBM (comparison #1)
- CatBoost (comparison #2)

Task:
- Default: **flat multiclass** (`Benign` + each `Attack Type`)
- Optional: **binary** (`Benign` vs `Malicious`)

Outputs:
- Trained model files per dataset/model
- Training summary JSON/CSV for thesis appendix


## 0) Environment & dependencies

In [1]:
# If needed:
# !pip install -U pandas numpy scikit-learn matplotlib tqdm xgboost lightgbm catboost joblib

import sys, platform
import pandas as pd
import numpy as np
from pathlib import Path
print("Python:", sys.version)
print("Platform:", platform.platform())

Python: 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Platform: Linux-6.8.0-49-generic-x86_64-with-glibc2.39


## 1) Configuration

### Important for comparability
To ensure all models see the same features across datasets, we align schemas:
- Either to a reference dataset (`FORCE_FEATURE_ORDER_FROM`)
- Or to the intersection of feature columns across all loaded datasets


In [ ]:
# ----------------------------
# Core task settings
# ----------------------------
TASK = "multiclass"   # "multiclass" or "binary"
LABEL_COL_MULTICLASS = "Attack Type"
LABEL_COL_BINARY     = "Label"
BENIGN_NAME          = "Benign"

# ----------------------------
# Training datasets
# ----------------------------
'''
TRAIN_DATASETS = {
    "mini5g_custom_train": "cedi-nidd/Flow-Dataset/merged_output.csv",
}'''

TRAIN_DATASETS = {
    # Example names — replace paths with your real train CSVs
    "5gnidd_ctgan_bal_aug_train": "Datasets/augmented_balanced_datasets/ctgan/mixed_augmented_balanced.csv",
    "5gnidd_ctgan_perclass_bal_aug_train":  "Datasets/augmented_per_class_datasets/ctgan/mixed_augmented_per_class_merged.csv",
    "5gnidd_tabddpm_bal_aug_train": "Datasets/augmented_balanced_datasets/tabddpm/mixed_augmented_balanced.csv",
    "5gnidd_tabddpm_perclass_bal_aug_train":  "Datasets/augmented_per_class_datasets/tabddpm/mixed_augmented_per_class_merged.csv",
    "5gnidd_not_aug_raw_train":           "Datasets/prepped_raw/train_input_with_label.csv",
    "5gnidd_not_aug_bal_raw_train":      "5G-NIDD_Aug_tabDDPM/Balanced/train_input_with_label.csv",
    "5gnidd_not_aug_dirty_raw_train":    "Datasets/Encoded_train_raw_160000.csv"
}

# ----------------------------
# Column hygiene
# ----------------------------
DROP_COLS = []  # e.g., ["Attack Tool"] 

# Schema alignment
FORCE_FEATURE_ORDER_FROM = None  # e.g., "5gnidd_tabddpm_bal_aug_train"

# ----------------------------
# Early stopping
# ----------------------------
USE_VAL_SPLIT = True
VAL_SIZE = 0.10
RANDOM_SEED = 42
N_JOBS = -1

# Speed knobs
FAST_DEV_RUN = False   # True = fewer trees/iterations (quick smoke test)
MAX_ROWS_PER_DATASET = None  # e.g., 300_000; None = use all

# Output
OUT_DIR = "classifier_training_gbdt"

# GPU toggle (requires GPU-enabled libs)
USE_GPU = True
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

print("Configured train datasets:", len(TRAIN_DATASETS))
print("OUT_DIR:", OUT_DIR)

Configured train datasets: 7
OUT_DIR: classifier_training_gbdt


## 2) Load datasets robustly + align feature schema

This section:
1) Loads each CSV robustly (skips malformed lines if any)
2) Applies `DROP_COLS`
3) Verifies the label column exists (based on `TASK`)
4) Aligns feature columns across all datasets to guarantee compatibility


In [3]:
from typing import Dict, Tuple
from collections import Counter
import joblib

def read_csv_robust(path: str) -> pd.DataFrame:
    return pd.read_csv(path, engine="python", on_bad_lines="skip")

def normalize_label_text(s: pd.Series) -> pd.Series:
    return s.astype(str).str.strip().replace({"nan":"Unknown", "None":"Unknown"})

def enforce_schema(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if DROP_COLS:
        df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors="ignore")
    return df

def prepare_xy(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.Series]:
    df = enforce_schema(df)

    if TASK == "multiclass":
        if LABEL_COL_MULTICLASS not in df.columns:
            raise ValueError(f"Missing multiclass label column '{LABEL_COL_MULTICLASS}'")
        y = normalize_label_text(df[LABEL_COL_MULTICLASS])
        X = df.drop(columns=[LABEL_COL_MULTICLASS], errors="ignore")
        return X, y

    if TASK == "binary":
        if LABEL_COL_BINARY not in df.columns:
            raise ValueError(f"Missing binary label column '{LABEL_COL_BINARY}'")
        y = normalize_label_text(df[LABEL_COL_BINARY])
        y01 = y.map({BENIGN_NAME: 0, "Malicious": 1, "malicious": 1, "BENIGN": 0}).fillna(y)
        X = df.drop(columns=[LABEL_COL_BINARY], errors="ignore")
        return X, y01

    raise ValueError("TASK must be 'multiclass' or 'binary'")

# Load
loaded: Dict[str, pd.DataFrame] = {}
for name, path in TRAIN_DATASETS.items():
    p = Path(path)
    if not p.exists():
        print(f"Missing dataset path for '{name}': {path}")
        continue
    df = read_csv_robust(str(p))
    if MAX_ROWS_PER_DATASET is not None and len(df) > MAX_ROWS_PER_DATASET:
        df = df.sample(MAX_ROWS_PER_DATASET, random_state=RANDOM_SEED).reset_index(drop=True)
    loaded[name] = df
    print(f"Loaded {name}: {df.shape}")

assert len(loaded) > 0, "No datasets loaded. Please set correct CSV paths in TRAIN_DATASETS."

# Align features
def align_features(dfs: Dict[str, pd.DataFrame]):
    keys = list(dfs.keys())

    if FORCE_FEATURE_ORDER_FROM and FORCE_FEATURE_ORDER_FROM in dfs:
        Xref, _ = prepare_xy(dfs[FORCE_FEATURE_ORDER_FROM])
        ref_cols = list(Xref.columns)
        print("Using reference schema from:", FORCE_FEATURE_ORDER_FROM, "n_features=", len(ref_cols))
    else:
        col_sets = []
        for k in keys:
            Xk, _ = prepare_xy(dfs[k])
            col_sets.append(set(Xk.columns))
        ref_cols = sorted(set.intersection(*col_sets))
        print("Using intersection schema. n_features=", len(ref_cols))

    aligned = {}
    for k in keys:
        Xk, yk = prepare_xy(dfs[k])
        common = [c for c in ref_cols if c in Xk.columns]
        Xk2 = Xk.loc[:, common].copy()
        Xk2 = Xk2.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
        df2 = Xk2.copy()
        if TASK == "multiclass":
            df2[LABEL_COL_MULTICLASS] = yk.values
        else:
            df2[LABEL_COL_BINARY] = yk.values
        aligned[k] = df2
        print(f"Aligned {k}: {df2.shape} (features={len(common)})")

    return aligned, ref_cols

aligned, FEATURE_ORDER = align_features(loaded)
print("Final feature count:", len(FEATURE_ORDER))

Loaded 5gnidd_ctgan_bal_aug_train: (320000, 41)
Loaded 5gnidd_ctgan_perclass_bal_aug_train: (320000, 42)
Loaded 5gnidd_tabddpm_bal_aug_train: (320000, 41)
Loaded 5gnidd_tabddpm_perclass_bal_aug_train: (320000, 42)
Loaded 5gnidd_not_aug_raw_train: (160000, 41)
Loaded 5gnidd_not_aug_bal_raw_train: (160000, 41)
Loaded 5gnidd_not_aug_dirty_raw_train: (160000, 96)
Using intersection schema. n_features= 40
Aligned 5gnidd_ctgan_bal_aug_train: (320000, 41) (features=40)
Aligned 5gnidd_ctgan_perclass_bal_aug_train: (320000, 41) (features=40)
Aligned 5gnidd_tabddpm_bal_aug_train: (320000, 41) (features=40)
Aligned 5gnidd_tabddpm_perclass_bal_aug_train: (320000, 41) (features=40)
Aligned 5gnidd_not_aug_raw_train: (160000, 41) (features=40)
Aligned 5gnidd_not_aug_bal_raw_train: (160000, 41) (features=40)
Aligned 5gnidd_not_aug_dirty_raw_train: (160000, 41) (features=40)
Final feature count: 40


## 3) Train the three GBDT models (training‑only)

We train each model on the full training dataset.

If `USE_VAL_SPLIT=True`, we hold out `VAL_SIZE` from the training data as a **validation set** for **early stopping**:
- This is best practice and does *not* replace an external test set.

We also compute **class weights** (inverse frequency) and use them as **sample weights** during training (best practice for imbalance).


In [4]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder
import time
from tqdm.auto import tqdm

# Optional imports
try:
    import xgboost as xgb
except Exception:
    xgb = None
    print("xgboost not importable. Install with: !pip install xgboost")

try:
    import lightgbm as lgb
    from lightgbm.basic import LightGBMError
except Exception:
    lgb = None
    LightGBMError = Exception
    print("lightgbm not importable. Install with: !pip install lightgbm")

try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None
    print("catboost not importable. Install with: !pip install catboost")

def get_class_weights(y: pd.Series):
    counts = Counter(y.astype(str))
    total = sum(counts.values())
    return {cls: total / (len(counts) * cnt) for cls, cnt in counts.items()}

def make_train_val(X: pd.DataFrame, y: pd.Series):
    if not USE_VAL_SPLIT:
        return X, y, None, None

    sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_SEED)
    tr_idx, val_idx = next(sss.split(X, y))
    return X.iloc[tr_idx], y.iloc[tr_idx], X.iloc[val_idx], y.iloc[val_idx]

def train_xgboost(Xtr, ytr, Xval, yval, w_tr, w_val):
    if xgb is None:
        return None, None
    n_estimators = 120 if FAST_DEV_RUN else 1200

    if TASK == "multiclass":
        model = xgb.XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            n_estimators=n_estimators,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_data_in_leaf=5,
            min_sum_hessian_in_leaf=1e-3,
            min_data_in_bin=1,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=RANDOM_SEED,
            device=("gpu" if USE_GPU else "cpu")
        )
    else:
        model = xgb.XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            n_estimators=n_estimators,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            min_data_in_leaf=5,
            min_sum_hessian_in_leaf=1e-3,
            min_data_in_bin=1,
            reg_lambda=1.0,
            tree_method="hist",
            n_jobs=N_JOBS,
            random_state=RANDOM_SEED,
            device=("gpu" if USE_GPU else "cpu")
        )

    if Xval is not None:
        model.fit(
            Xtr, ytr,
            sample_weight=w_tr,
            eval_set=[(Xval, yval)],
            sample_weight_eval_set=[w_val],
            verbose=False
        )
        best = int(getattr(model, "best_iteration", n_estimators))
    else:
        model.fit(Xtr, ytr, sample_weight=w_tr, verbose=False)
        best = n_estimators

    return model, best

def train_lightgbm(Xtr, ytr, Xval, yval, w_tr, w_val):
    if lgb is None:
        return None, None
    n_estimators = 200 if FAST_DEV_RUN else 4000

    def make_model(device):
        if TASK == "multiclass":
            return lgb.LGBMClassifier(
                objective="multiclass",
                n_estimators=n_estimators,
                learning_rate=0.05,
                num_leaves=63,
                subsample=0.8,
                colsample_bytree=0.8,
                min_data_in_leaf=5,
                min_sum_hessian_in_leaf=1e-3,
                min_data_in_bin=1,
                reg_lambda=1.0,
                n_jobs=N_JOBS,
                random_state=RANDOM_SEED,
                device=device
            )
        else:
            return lgb.LGBMClassifier(
                objective="binary",
                n_estimators=n_estimators,
                learning_rate=0.05,
                num_leaves=63,
                subsample=0.8,
                colsample_bytree=0.8,
                min_data_in_leaf=5,
                min_sum_hessian_in_leaf=1e-3,
                min_data_in_bin=1,
                reg_lambda=1.0,
                n_jobs=N_JOBS,
                random_state=RANDOM_SEED,
                device=device
            )

    def fit_model(model):
        if Xval is not None:
            model.fit(
                Xtr, ytr,
                sample_weight=w_tr,
                eval_set=[(Xval, yval)],
                eval_sample_weight=[w_val],
                callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
            )
            return int(getattr(model, "best_iteration_", n_estimators))
        model.fit(Xtr, ytr, sample_weight=w_tr)
        return n_estimators

    model = make_model("gpu" if USE_GPU else "cpu")
    try:
        best = fit_model(model)
    except Exception as e:
        if USE_GPU:
            print("LightGBM GPU failed, retrying on CPU:", repr(e))
            model = make_model("cpu")
            best = fit_model(model)
        else:
            raise

    return model, best

def train_catboost(Xtr, ytr, Xval, yval, w_tr):
    if CatBoostClassifier is None:
        return None, None
    iters = 400 if FAST_DEV_RUN else 8000
    loss_fn = "MultiClass" if TASK == "multiclass" else "Logloss"

    model = CatBoostClassifier(
        loss_function=loss_fn,
        iterations=iters,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=3.0,
        random_seed=RANDOM_SEED,
        task_type=("GPU" if USE_GPU else "CPU"),
        verbose=False
    )
    if Xval is not None:
        model.fit(Xtr, ytr, sample_weight=w_tr, eval_set=(Xval, yval), use_best_model=True)
        best = int(getattr(model, "get_best_iteration", lambda: iters)())
    else:
        model.fit(Xtr, ytr, sample_weight=w_tr)
        best = iters
    return model, best

print("Ready to train. USE_VAL_SPLIT =", USE_VAL_SPLIT, "VAL_SIZE =", VAL_SIZE)

Ready to train. USE_VAL_SPLIT = True VAL_SIZE = 0.1


## 4) Train on all training datasets + export models & training summaries

For each dataset:
1) Builds X/y
2) Computes class weights → sample weights
3) Optionally makes a small validation split (for early stopping)
4) Trains XGBoost / LightGBM / CatBoost
5) Saves model artifacts to disk
6) Writes a training summary JSON for thesis traceability


In [5]:
import json

train_summaries = {}
for ds_name, df in aligned.items():
    print("\n" + "="*80)
    print("TRAIN DATASET:", ds_name, "shape:", df.shape)

    if TASK == "multiclass":
        y = normalize_label_text(df[LABEL_COL_MULTICLASS])
        X = df.drop(columns=[LABEL_COL_MULTICLASS], errors="ignore")
    else:
        y = normalize_label_text(df[LABEL_COL_BINARY])
        X = df.drop(columns=[LABEL_COL_BINARY], errors="ignore")

    # sample weights
    cw = get_class_weights(y)
    w = y.astype(str).map(cw).values

    # train/val
    Xtr, ytr, Xval, yval = make_train_val(X, y)
    if Xval is not None:
        w_tr = ytr.astype(str).map(cw).values
        w_val = yval.astype(str).map(cw).values
        print("Using validation split:", Xval.shape)
    else:
        w_tr = w
        w_val = None
        print("No validation split. Training on full dataset.")

    ds_out = Path(OUT_DIR) / ds_name
    ds_out.mkdir(parents=True, exist_ok=True)

    # label encoding for XGBoost (multiclass only)
    le = None
    ytr_xgb = ytr
    yval_xgb = yval
    if TASK == "multiclass":
        le = LabelEncoder()
        le.fit(y.astype(str))
        ytr_xgb = le.transform(ytr.astype(str))
        if yval is not None:
            yval_xgb = le.transform(yval.astype(str))

    # Train models
    ds_summary = {
        "dataset": ds_name,
        "task": TASK,
        "label_col": LABEL_COL_MULTICLASS if TASK=="multiclass" else LABEL_COL_BINARY,
        "rows": int(len(df)),
        "n_features": int(len(FEATURE_ORDER)),
        "use_val_split": bool(USE_VAL_SPLIT),
        "val_size": float(VAL_SIZE) if USE_VAL_SPLIT else None,
        "class_distribution": dict(Counter(y.astype(str))),
        "label_mapping": {str(lbl): int(i) for i, lbl in enumerate(le.classes_)} if le is not None else None,
        "models": {}
    }

    # XGBoost
    t0 = time.time()
    model, best = train_xgboost(Xtr, ytr_xgb, Xval, yval_xgb, w_tr, w_val)
    if model is not None:
        model_path = ds_out / "xgboost_model.json"
        model.get_booster().save_model(str(model_path))
        ds_summary["models"]["xgboost"] = {
            "model_path": str(model_path),
            "train_time_sec": float(time.time() - t0),
            "best_iteration": int(best)
        }
        print("Saved XGBoost:", model_path)

    # LightGBM
    t0 = time.time()
    model, best = train_lightgbm(Xtr, ytr, Xval, yval, w_tr, w_val)
    if model is not None:
        model_path = ds_out / "lightgbm_model.txt"
        model.booster_.save_model(str(model_path))
        ds_summary["models"]["lightgbm"] = {
            "model_path": str(model_path),
            "train_time_sec": float(time.time() - t0),
            "best_iteration": int(best)
        }
        print("Saved LightGBM:", model_path)

    # CatBoost
    t0 = time.time()
    model, best = train_catboost(Xtr, ytr, Xval, yval, w_tr)
    if model is not None:
        model_path = ds_out / "catboost_model.cbm"
        model.save_model(str(model_path))
        ds_summary["models"]["catboost"] = {
            "model_path": str(model_path),
            "train_time_sec": float(time.time() - t0),
            "best_iteration": int(best) if best is not None else None
        }
        print("Saved CatBoost:", model_path)

    # Save per-dataset summary
    summary_path = ds_out / "training_summary.json"
    with open(summary_path, "w") as f:
        json.dump(ds_summary, f, indent=2)
    print("Wrote:", summary_path)

    train_summaries[ds_name] = ds_summary

# Global summary
global_path = Path(OUT_DIR) / "training_summaries_all.json"
with open(global_path, "w") as f:
    json.dump(train_summaries, f, indent=2)
print("\nWrote global summary:", global_path)

# Quick table
rows = []
for ds, s in train_summaries.items():
    for m, md in s["models"].items():
        rows.append({
            "dataset": ds,
            "model": m,
            "rows": s["rows"],
            "n_features": s["n_features"],
            "train_time_sec": md.get("train_time_sec"),
            "best_iteration": md.get("best_iteration")
        })
df_sum = pd.DataFrame(rows).sort_values(["dataset","model"])
display(df_sum)


TRAIN DATASET: 5gnidd_ctgan_bal_aug_train shape: (320000, 41)
Using validation split: (32000, 40)


/home/ubuntu/ganenv/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:29:45] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "min_data_in_bin", "min_data_in_leaf", "min_sum_hessian_in_leaf" } are not used.

  warnings.warn(smsg, UserWarning)


Saved XGBoost: classifier_training_gbdt/5gnidd_ctgan_bal_aug_train/xgboost_model.json
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2610
[LightGBM] [Info] Number of data points in the train set: 288000, number of used features: 40
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram

/home/ubuntu/ganenv/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:33:50] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "min_data_in_bin", "min_data_in_leaf", "min_sum_hessian_in_leaf" } are not used.

  warnings.warn(smsg, UserWarning)


Saved XGBoost: classifier_training_gbdt/5gnidd_ctgan_perclass_bal_aug_train/xgboost_model.json
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2610
[LightGBM] [Info] Number of data points in the train set: 288000, number of used features: 40
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of 

...

/home/ubuntu/ganenv/lib/python3.12/site-packages/xgboost/core.py:158: UserWarning: [15:51:47] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "min_data_in_bin", "min_data_in_leaf", "min_sum_hessian_in_leaf" } are not used.

  warnings.warn(smsg, UserWarning)


Saved XGBoost: classifier_training_gbdt/5gnidd_not_aug_dirty_raw_train/xgboost_model.json
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 2351
[LightGBM] [Info] Number of data points in the train set: 144000, number of used features: 38
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...
[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histo

[LightGBM] [Fatal] Check failed: (best_split_info.left_count) > (0) at /__w/1/s/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 852 .



[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007358 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2351
[LightGBM] [Info] Number of data points in the train set: 144000, number of used features: 38
[LightGBM] [Warning] min_sum_hessian_in_leaf is set=0.001, min_child_weight=0.001 will be ignored. Current value: min_sum_hessian_in_leaf=0.001
[LightGBM] [Warning] min_data_in_leaf is set=5, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=5
[LightGBM] [Info] Start training from score -2.197386
[LightGBM] [Info] Start training from score -2.197378
[LightGBM]

,dataset,model,rows,n_features,train_time_sec,best_iteration
2,5gnidd_ctgan_bal_aug_train,catboost,320000,40,165.170608,7999
1,5gnidd_ctgan_bal_aug_train,lightgbm,320000,40,29.974987,160
0,5gnidd_ctgan_bal_aug_train,xgboost,320000,40,49.967004,1200
5,5gnidd_ctgan_perclass_bal_aug_train,catboost,320000,40,148.645258,7999
4,5gnidd_ctgan_perclass_bal_aug_train,lightgbm,320000,40,133.084678,3045
3,5gnidd_ctgan_perclass_bal_aug_train,xgboost,320000,40,25.083505,1200
17,5gnidd_not_aug_bal_raw_train,catboost,160000,40,115.069818,7996
16,5gnidd_not_aug_bal_raw_train,lightgbm,160000,40,8.438064,141
15,5gnidd_not_aug_bal_raw_train,xgboost,160000,40,25.126271,1200
20,5gnidd_not_aug_dirty_raw_train,catboost,160000,40,117.609770,5979
